# Data Battle 2026 – Météorage
## Prédiction de la fin d'un orage — Notebook complet
---
**Objectif** : Estimer la probabilité qu'un éclair nuage-sol soit le **dernier de l'alerte**,  
afin de permettre aux aéroports de reprendre leurs opérations le plus tôt possible.

| Métrique | Valeur |
|---|---|
| AUC (cross-val OOF) | **0.9197 ± 0.0040** |
| Accuracy max | **95.4%** |
| Baseline météorage | 80.77% |
| Gain (eval, θ=0.95) | **80.4h** |
| Risque (eval, θ=0.95) | **0.95%** < 2% ✓ |


## 1. Imports et chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
import pickle, os, warnings
warnings.filterwarnings('ignore')

os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

TRAIN_PATH = 'segment_alerts_all_airports_train/segment_alerts_all_airports_train.csv'
EVAL_PATH  = 'segment_alerts_all_airports_eval.csv'

df_train = pd.read_csv(TRAIN_PATH)
df_train['date'] = pd.to_datetime(df_train['date'], utc=True)
df_eval  = pd.read_csv(EVAL_PATH)
df_eval['date']  = pd.to_datetime(df_eval['date'],  utc=True)

print(f"Train : {df_train.shape}  |  Eval : {df_eval.shape}")
print(f"Aéroports : {df_train['airport'].unique()}")
print(f"Train période : {df_train['date'].min().date()} → {df_train['date'].max().date()}")
print(f"Eval  période : {df_eval['date'].min().date()} → {df_eval['date'].max().date()}")
df_train.head()

## 2. Exploration des données (EDA)

In [ ]:
# Statistiques générales
print('=== TRAIN ===')
print(f"Éclairs totaux: {len(df_train):,}")
labeled = df_train[df_train['airport_alert_id'].notnull()]
print(f"Éclairs en alerte: {len(labeled):,} ({len(labeled)/len(df_train)*100:.1f}%)")
print(f"Nb alertes: {labeled['airport_alert_id'].nunique()}")
print(f"Distribution cible:\n{labeled['is_last_lightning_cloud_ground'].value_counts(dropna=False)}")

In [ ]:
# Visualisation EDA - voir plots/ pour les figures générées
# Charger les figures sauvegardées
figs = ['01_airport_analysis', '02_temporal_analysis', '03_alert_analysis',
        '04_spatial_analysis', '05_last_lightning_analysis', '06_correlation_matrix']
        
for fname in figs[:2]:
    from IPython.display import Image, display
    if os.path.exists(f'plots/{fname}.png'):
        display(Image(f'plots/{fname}.png'))

## 3. Feature Engineering

### Hypothèses testées :
1. **Les derniers éclairs arrivent plus espacés** → features inter-arrival
2. **L'orage s'éloigne vers la fin** → tendance de distance
3. **L'activité diminue** → densité temporelle décroissante
4. **Position dans l'alerte** → rang et temps depuis le début

> ⚠️ **Important** : on n'utilise que des features **causales** (passé connu au moment t).  
> Features exclues : `total_in_alert`, `rel_pos`, `remaining` (fuite de données !)


In [ ]:
def make_features(df_cg, le=None, fit_le=False):
    """Feature engineering causal sur les éclairs CG d'alertes."""
    df_cg = df_cg.sort_values(['airport', 'airport_alert_id', 'date']).reset_index(drop=True)
    grp = df_cg.groupby(['airport', 'airport_alert_id'])
    
    # Position dans l'alerte (nb d'éclairs CG observés jusqu'ici)
    df_cg['rank_cg'] = grp.cumcount()
    alert_start = grp['date'].transform('min')
    df_cg['t_since_start_s'] = (df_cg['date'] - alert_start).dt.total_seconds()
    
    # Temps inter-éclairs
    df_cg['dt_prev_s'] = grp['date'].diff().dt.total_seconds().fillna(0)
    
    # Features glissantes sur les W éclairs précédents
    for w in [3, 5, 10]:
        df_cg[f'dt_mean_{w}']    = grp['dt_prev_s'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df_cg[f'dt_max_{w}']     = grp['dt_prev_s'].transform(lambda x: x.rolling(w, min_periods=1).max())
        df_cg[f'dt_std_{w}']     = grp['dt_prev_s'].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
        df_cg[f'dist_mean_{w}']  = grp['dist'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df_cg[f'dist_trend_{w}'] = grp['dist'].transform(
            lambda x: x.rolling(w, min_periods=2).apply(
                lambda v: v.iloc[-1]-v.iloc[0] if len(v)>1 else 0, raw=False))
        df_cg[f'amp_mean_{w}']   = grp['amplitude'].transform(lambda x: x.abs().rolling(w, min_periods=1).mean())
        df_cg[f'amp_std_{w}']    = grp['amplitude'].transform(lambda x: x.abs().rolling(w, min_periods=1).std().fillna(0))
    
    # Densité d'activité dans fenêtre temporelle glissante
    def rolling_time_count(group_dates, window_s):
        dates = group_dates.values
        result = np.zeros(len(dates))
        for i in range(len(dates)):
            cutoff = dates[i] - np.timedelta64(int(window_s*1e9), 'ns')
            result[i] = np.sum(dates[:i+1] >= cutoff)
        return pd.Series(result, index=group_dates.index)
    
    for w_min, w_s in [(2,120),(5,300),(10,600),(15,900),(30,1800)]:
        df_cg[f'n_prev_{w_min}min'] = grp['date'].transform(lambda x: rolling_time_count(x, w_s))
    
    df_cg['rate_decline'] = df_cg['n_prev_5min'] / (df_cg['n_prev_10min'] + 1)
    
    # Features physiques
    df_cg['amp_abs']  = df_cg['amplitude'].abs()
    df_cg['amp_sign'] = np.sign(df_cg['amplitude'])
    df_cg['month']    = df_cg['date'].dt.month
    df_cg['hour']     = df_cg['date'].dt.hour
    
    # Encodage aéroport
    if le is None or fit_le:
        le = LabelEncoder()
        df_cg['airport_enc'] = le.fit_transform(df_cg['airport'])
    else:
        df_cg['airport_enc'] = le.transform(df_cg['airport'])
    
    return df_cg, le

# Préparer les données train labellisées
df_cg_tr = df_train[df_train['airport_alert_id'].notnull()].copy()
df_cg_tr['is_last'] = df_cg_tr['is_last_lightning_cloud_ground'].astype(bool)
df_cg_tr, le = make_features(df_cg_tr, fit_le=True)
print(f"Shape features: {df_cg_tr.shape}")
print(f"Classes: {df_cg_tr['is_last'].value_counts().to_dict()}")

## 4. Entraînement LightGBM (GroupKFold)

In [ ]:
FEATURE_COLS = [
    'rank_cg', 't_since_start_s', 'dt_prev_s',
    'dt_mean_3', 'dt_mean_5', 'dt_mean_10',
    'dt_max_3',  'dt_max_5',  'dt_max_10',
    'dt_std_3',  'dt_std_5',  'dt_std_10',
    'dist', 'dist_mean_3', 'dist_mean_5', 'dist_mean_10',
    'dist_trend_3', 'dist_trend_5', 'dist_trend_10',
    'amp_abs', 'amp_sign', 'maxis',
    'amp_mean_3', 'amp_mean_5', 'amp_mean_10',
    'amp_std_3',
    'n_prev_2min', 'n_prev_5min', 'n_prev_10min', 'n_prev_15min', 'n_prev_30min',
    'rate_decline',
    'azimuth', 'month', 'hour', 'airport_enc',
]

X = df_cg_tr[FEATURE_COLS].astype(float)
y = df_cg_tr['is_last'].astype(int)
groups = df_cg_tr['airport_alert_id']

# Cross-validation GroupKFold (alertes non partagées entre folds)
gkf = GroupKFold(n_splits=5)
oof_probs = np.zeros(len(y))
fi_arr = np.zeros(len(FEATURE_COLS))
fold_aucs = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    mdl = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                          num_leaves=63, class_weight='balanced', 
                          subsample=0.8, colsample_bytree=0.8, 
                          random_state=42, n_jobs=-1, verbose=-1)
    mdl.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[val_idx], y.iloc[val_idx])])
    oof_probs[val_idx] = mdl.predict_proba(X.iloc[val_idx])[:, 1]
    fi_arr += mdl.feature_importances_
    auc = roc_auc_score(y.iloc[val_idx], oof_probs[val_idx])
    fold_aucs.append(auc)
    print(f"  Fold {fold+1}: AUC={auc:.4f}")

print(f"\nAUC global: {roc_auc_score(y, oof_probs):.4f} ± {np.std(fold_aucs):.4f}")

In [ ]:
# Entraînement final sur 100% des données
final_model = LGBMClassifier(n_estimators=700, learning_rate=0.05, max_depth=6,
                              num_leaves=63, class_weight='balanced',
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=42, n_jobs=-1, verbose=-1)
final_model.fit(X, y)

# Sauvegarder
pickle.dump({'model': final_model, 'le': le, 'features': FEATURE_COLS}, 
            open('models/lgbm_v2.pkl', 'wb'))
print("Modèle sauvegardé ✓")

# Feature importance
fi_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': fi_arr/5})
fi_df = fi_df.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
fi_top = fi_df.head(20)
ax.barh(range(len(fi_top)), fi_top['importance'], color='steelblue')
ax.set_yticks(range(len(fi_top)))
ax.set_yticklabels(fi_top['feature'])
ax.invert_yaxis()
ax.set_title('Top 20 Features')
plt.tight_layout()
plt.show()

## 5. Génération des prédictions et évaluation

In [ ]:
def generate_predictions(df_cg, model, le, feature_cols):
    """Génère le fichier predictions.csv dans le format requis."""
    df_feat, _ = make_features(df_cg.copy(), le=le)
    probs = model.predict_proba(df_feat[feature_cols].astype(float))[:, 1]
    df_feat['confidence'] = probs
    return pd.DataFrame({
        'airport': df_feat['airport'],
        'airport_alert_id': df_feat['airport_alert_id'],
        'prediction_date': df_feat['date'],
        'predicted_date_end_alert': df_feat['date'],
        'confidence': df_feat['confidence'],
    })

# Prédictions eval
df_cg_ev = df_eval[df_eval['airport_alert_id'].notnull()].copy()
preds_eval = generate_predictions(df_cg_ev, final_model, le, FEATURE_COLS)
preds_eval.to_csv('predictions_eval.csv', index=False)
print(f"predictions_eval.csv: {len(preds_eval):,} prédictions, {preds_eval['airport_alert_id'].nunique()} alertes")
preds_eval.head()

In [ ]:
# Évaluation complète avec protocole officiel
def evaluate_predictions(preds_csv_path, df_full, acceptable_risk=0.02):
    MAX_GAP = 30
    min_dist = 3
    tot_l = len(df_full[df_full['dist'] < min_dist])
    alerts = df_full.groupby(['airport','airport_alert_id'])
    preds = pd.read_csv(preds_csv_path)
    preds['predicted_date_end_alert'] = pd.to_datetime(preds['predicted_date_end_alert'], utc=True)
    
    thetas = [i/20 for i in range(21)]
    results = {}
    for theta in thetas:
        pt = preds[preds['confidence'] >= theta]
        if len(pt) == 0: continue
        pm = pt.groupby(['airport','airport_alert_id'])['predicted_date_end_alert'].min()
        gain, missed = 0, 0
        for (ap, aid), ep in pm.items():
            l = alerts.get_group((ap, aid))
            eb = pd.to_datetime(l['date'], utc=True).max() + pd.Timedelta(minutes=MAX_GAP)
            gain += (eb - ep).total_seconds()
            missed += sum(pd.to_datetime(l[l['dist']<min_dist]['date'], utc=True) > ep)
        results[theta] = (gain/3600, missed/tot_l if tot_l>0 else 0)
    
    valid = [(g, t, r) for t,(g,r) in results.items() if r < acceptable_risk]
    best = max(valid) if valid else None
    return results, best, tot_l

results_eval, best_eval, tot_l = evaluate_predictions('predictions_eval.csv', df_eval)

print("Theta  | Gain (h) | Risk")
for t in [0.5, 0.7, 0.8, 0.85, 0.9, 0.95]:
    if t in results_eval:
        g, r = results_eval[t]
        flag = "✓" if r < 0.02 else "✗"
        print(f"  {t:.2f}  | {g:8.1f} | {r:.4f} {flag}")

if best_eval:
    print(f"\n=> Meilleur theta (R<2%): θ={best_eval[1]:.2f}, Gain={best_eval[0]:.1f}h, Risk={best_eval[2]:.4f}")

In [ ]:
# Courbe Gain vs Risque
thetas_list = sorted(results_eval.keys())
gains  = [results_eval[t][0] for t in thetas_list]
risks  = [results_eval[t][1] for t in thetas_list]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sc = ax.scatter(risks, gains, c=thetas_list, cmap='viridis', s=80, zorder=5)
ax.plot(risks, gains, 'gray', alpha=0.4)
for t,g,r in zip(thetas_list,gains,risks):
    if t in [0.3, 0.5, 0.7, 0.85, 0.9, 0.95]:
        ax.annotate(f'θ={t:.2f}', (r,g), textcoords='offset points', xytext=(5,3), fontsize=9)
ax.axvline(x=0.02, color='red', linestyle='--', label='R_accept=2%')
ax.set_xlabel('Risque (éclairs dangereux manqués)')
ax.set_ylabel('Gain de temps (heures)')
ax.set_title("Courbe Gain vs Risque — Eval")
ax.legend(); plt.colorbar(sc, ax=ax, label='Theta')

ax = axes[1]
prec, rec, _ = precision_recall_curve(
    df_cg_tr['is_last'].astype(int), 
    final_model.predict_proba(df_cg_tr[FEATURE_COLS].astype(float))[:,1])
ax.plot(rec, prec, 'b-', linewidth=2)
ax.fill_between(rec, prec, alpha=0.15)
ap = average_precision_score(df_cg_tr['is_last'].astype(int), 
                              final_model.predict_proba(df_cg_tr[FEATURE_COLS].astype(float))[:,1])
ax.set_title(f'Precision-Recall (AP={ap:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.axhline(y=df_cg_tr['is_last'].mean(), color='red', linestyle='--', label='Baseline')
ax.legend()

plt.tight_layout()
plt.savefig('plots/12_final_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Analyse des spécificités par aéroport

Chaque aéroport a des conditions géographiques et météorologiques différentes.


In [ ]:
# Performance par aéroport
df_cg_tr['oof_prob'] = oof_probs  # depuis la cross-val

for airport in df_cg_tr['airport'].unique():
    sub = df_cg_tr[df_cg_tr['airport'] == airport]
    auc = roc_auc_score(sub['is_last'].astype(int), sub['oof_prob'])
    nb_alerts = sub['airport_alert_id'].nunique()
    moy_eclairs = len(sub) / nb_alerts
    print(f"{airport:10s}: AUC={auc:.4f} | {nb_alerts} alertes | {moy_eclairs:.0f} éclairs/alerte")

## 7. Conclusions et perspectives

### Résultats principaux
- **AUC = 0.92** sur données d'entraînement (cross-validation par alerte)  
- **Gain = 80.4h** sur le set d'évaluation avec risque < 1% (θ=0.95)  
- **Dépasse la baseline** météorage de 80.77% d'accuracy  

### Features les plus importantes
1. **Azimuth** — direction de l'orage par rapport à l'aéroport
2. **Temps inter-éclairs** (dt_prev_s) — l'espacement augmente vers la fin  
3. **Maxis** — intensité du courant  
4. **Tendance de distance** (dist_trend) — l'orage s'éloigne  
5. **Rang dans l'alerte** (rank_cg) et **temps depuis le début**  

### Améliorations possibles
- Intégrer les éclairs **intra-nuage** comme contexte  
- Modèles par aéroport ou features géographiques  
- Modèles séquentiels (LSTM, Transformer)  
- Features météo (vent, pression)
